In [ ]:
#@title Clone GitHub repo (idempotent)
import os, sys, subprocess

REPO_URL = "https://github.com/dauparas/ProteinMPNN.git"
REPO_DIR = "ProteinMPNN"
BRANCH   = "main"  # 可改为需要的分支

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return r

if not os.path.isdir(REPO_DIR):
    # 第一次：浅克隆，加快 Colab 速度
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    # 之后：获取最新并对齐到远端 main（避免本地脏状态影响）
    run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

# 打印当前 commit，便于追溯
run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)

# 加入 Python 路径
if f"/content/{REPO_DIR}" not in sys.path:
    sys.path.append(f"/content/{REPO_DIR}")

print("✅ ProteinMPNN repo is ready and on sys.path.")

In [ ]:
#@title Setup Model
import os, shutil, copy, random, warnings, os.path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import matplotlib.pyplot as plt

# ProteinMPNN utilities (repo was added to sys.path in Module 1)
from protein_mpnn_utils import (
    loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t,
    cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB,
    StructureDataset, StructureDatasetPDB, ProteinMPNN
)

# Select device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🟢 Torch device: {device} | CUDA available: {torch.cuda.is_available()}")

# v_48_xxx are trained with different edge counts/noise
model_name = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]
backbone_noise = 0.00   # Standard deviation of Gaussian noise to add to backbone atoms

# Paths (clone step put repo at /content/ProteinMPNN)
path_to_model_weights = "/content/ProteinMPNN/vanilla_model_weights"
hidden_dim = 128
num_layers = 3

# Resolve checkpoint path
model_folder_path = path_to_model_weights if path_to_model_weights.endswith("/") else path_to_model_weights + "/"
checkpoint_path = os.path.join(model_folder_path, f"{model_name}.pt")

# Sanity check for weights
if not os.path.isdir(path_to_model_weights):
    raise FileNotFoundError(
        f"Model weights folder not found: {path_to_model_weights}\n"
        "Make sure Module 1 (clone) ran successfully, and the repository includes 'vanilla_model_weights'."
    )
if not os.path.isfile(checkpoint_path):
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}\n"
        "Available files:\n  " + "\n  ".join(sorted(os.listdir(path_to_model_weights)))
    )

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
print("Number of edges in checkpoint:", checkpoint.get("num_edges", "N/A"))
print(f"Training noise level: {checkpoint.get('noise_level', 'N/A')} Å")

# Build model
model = ProteinMPNN(
    num_letters=21,
    node_features=hidden_dim,
    edge_features=hidden_dim,
    hidden_dim=hidden_dim,
    num_encoder_layers=num_layers,
    num_decoder_layers=num_layers,
    augment_eps=backbone_noise,
    k_neighbors=checkpoint.get("num_edges", 48),  # default to 48 if key missing
)
model.to(device)

# Load weights
missing, unexpected = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
if missing:
    print("⚠️ Missing keys in state_dict:", missing)
if unexpected:
    print("⚠️ Unexpected keys in state_dict:", unexpected)

model.eval()
print("✅ Model loaded and set to eval()")

In [ ]:
#@title Helper functions
def make_tied_positions_for_homomers(pdb_dict_list):
    """
    Create tied positions dictionary for homomeric complexes.
    This ensures that identical chains (A, B, C, ...) have identical designed residues.

    Args:
        pdb_dict_list (list): List of parsed PDB dictionaries from parse_PDB().

    Returns:
        dict: Mapping {pdb_name: [{chainA:[i], chainB:[i], ...}, ...]}.
    """
    my_dict = {}
    for result in pdb_dict_list:
        # find all chain IDs like A, B, C
        all_chain_list = sorted(
            [item[-1:] for item in list(result) if item.startswith('seq_chain')]
        )
        if not all_chain_list:
            continue
        chain_length = len(result[f"seq_chain_{all_chain_list[0]}"])
        tied_positions_list = []
        for i in range(1, chain_length + 1):
            temp_dict = {}
            for chain in all_chain_list:
                temp_dict[chain] = [i]
            tied_positions_list.append(temp_dict)
        my_dict[result['name']] = tied_positions_list
    return my_dict

print("✅ Helper function 'make_tied_positions_for_homomers' is ready.")

In [ ]:
#@title Inputs & preprocessing (use params from Cell 1; fallback to defaults)
import os, re, numpy as np
from google.colab import files

# ---------- helpers ----------
def get_pdb(pdb_code=""):
  """If pdb_code empty -> prompt upload; else wget from RCSB and return path"""
  if pdb_code is None or str(pdb_code).strip() == "":
    upload_dict = files.upload()
    pdb_bytes = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_bytes)
    return "tmp.pdb"
  else:
    code = str(pdb_code).strip().upper()
    os.system(f"wget -qnc https://files.rcsb.org/view/{code}.pdb")
    return f"{code}.pdb"

def _spaced_to_list(spaced):
  if not spaced: return []
  # "A B C" / "A,B" / "A;B" -> ["A","B","C"]
  return [x for x in re.sub(r"[^A-Za-z]+", ",", str(spaced)).split(",") if x]

# ---------- source parameters ----------
# Prefer variables produced by the first cell; otherwise fallback to your original defaults.
pdb            = globals().get("pdb", "1O91")
homomer        = bool(globals().get("homomer", True))
designed_chain = globals().get("designed_chain", "A B C")
fixed_chain    = globals().get("fixed_chain", "")
num_seqs       = int(globals().get("num_seqs", 1))
sampling_temp  = str(globals().get("sampling_temp", "0.1"))  # keep string style consistent with legacy

# ---------- build derived variables ----------
pdb_path = get_pdb(pdb)

# lists for chain specification
designed_chain_list = _spaced_to_list(designed_chain) if designed_chain else []
fixed_chain_list    = _spaced_to_list(fixed_chain)    if fixed_chain    else []

# union set (may be empty if user leaves both blank)
chain_list = sorted(list(set(designed_chain_list + fixed_chain_list)))

# other knobs (unchanged from your original)
save_score=0
save_probs=0
score_only=0
conditional_probs_only=0
conditional_probs_only_backbone=0

batch_size=1
max_length=20000

out_folder='.'
jsonl_path=''
omit_AAs='X'

pssm_multi=0.0
pssm_threshold=0.0
pssm_log_odds_flag=0
pssm_bias_flag=0

folder_for_outputs = out_folder

# from string like "0.1" or "0.1 0.3" to list of floats
temperatures = [float(item) for item in sampling_temp.split()]
omit_AAs_list = omit_AAs
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

chain_id_dict = None
fixed_positions_dict = None
pssm_dict = None
omit_AA_dict = None
bias_AA_dict = None
tied_positions_dict = None
bias_by_res_dict = None
bias_AAs_np = np.zeros(len(alphabet))

# ---------- parse PDB & dataset ----------
# If user did not specify any chain, parse all chains (input_chain_list=None)
_input_chains = chain_list if len(chain_list) > 0 else None
pdb_dict_list = parse_PDB(pdb_path, input_chain_list=_input_chains)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

# if chain_list was empty, infer all chains from parsed PDB for printing and downstream mapping
if not chain_list:
  keys = [k for k in pdb_dict_list[0].keys() if k.startswith("seq_chain_")]
  chain_list = sorted([k.replace("seq_chain_", "") for k in keys])

# mapping for ProteinMPNN runner: {pdb_name: (design_list, fixed_list)}
chain_id_dict = {pdb_dict_list[0]['name']: (designed_chain_list, fixed_chain_list)}

# info prints
print("=== Effective Params (after UI/URL merge) ===")
print({
  "pdb": pdb, "pdb_path": pdb_path,
  "homomer": homomer,
  "designed_chain": designed_chain, "designed_chain_list": designed_chain_list,
  "fixed_chain": fixed_chain,       "fixed_chain_list": fixed_chain_list,
  "chain_list": chain_list,
  "num_seqs": num_seqs,
  "temperatures": temperatures,
})
print("============================================")

for chain in chain_list:
  l = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  print(f"Length of chain {chain}: {l}")

# ---------- tied positions for homomers ----------
if homomer:
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
else:
  tied_positions_dict = None

In [ ]:
#@title RUN
import os, copy, numpy as np, torch
from datetime import datetime

assert 'dataset_valid' in globals() and len(dataset_valid) > 0, "dataset_valid is empty. Check Module 4."
assert 'model' in globals(), "Model is not loaded. Run Module 2."
assert 'temperatures' in globals() and len(temperatures) > 0, "No temperatures. Check Module 4."
assert 'NUM_BATCHES' in globals() and 'BATCH_COPIES' in globals(), "NUM_BATCHES/BATCH_COPIES not set."

out_dir = os.path.join(folder_for_outputs, f"mpnn_outputs_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(out_dir, exist_ok=True)

fasta_path = os.path.join(out_dir, "designed_sequences.fasta")      # ← 可选保存
tsv_path   = os.path.join(out_dir, "samples.tsv")                   # ← 可选保存
print(f"Outputs will be saved under: {out_dir}")

all_probs_list, all_log_probs_list, S_sample_list = [], [], []

with torch.no_grad():
    print('Generating sequences...')
    for ix, protein in enumerate(dataset_valid):
        score_list = []
        batch_clones = [copy.deepcopy(protein) for _ in range(BATCH_COPIES)]
        X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
        visible_list_list, masked_list_list, masked_chain_length_list_list, \
        chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, \
        tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, \
        bias_by_res_all, tied_beta = tied_featurize(
            batch_clones, device, chain_id_dict, fixed_positions_dict,
            omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict
        )

        pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float()
        name_ = batch_clones[0]['name']

        # native score for reference
        randn_1 = torch.randn(chain_M.shape, device=X.device)
        log_probs_nat = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1)
        mask_for_loss = mask*chain_M*chain_M_pos
        native_score = _scores(S, log_probs_nat, mask_for_loss).cpu().data.numpy()

        # optional: open files once
        fasta_fh = open(fasta_path, "a")   # ← 可选保存
        tsv_fh   = open(tsv_path, "a")     # ← 可选保存

        for temp in temperatures:
            for j in range(NUM_BATCHES):
                randn_2 = torch.randn(chain_M.shape, device=X.device)
                if tied_positions_dict is None:
                    sample_dict = model.sample(
                        X, randn_2, S, chain_M, chain_encoding_all, residue_idx,
                        mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np,
                        chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask,
                        pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi,
                        pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask,
                        pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all
                    )
                    S_sample = sample_dict["S"]
                else:
                    sample_dict = model.tied_sample(
                        X, randn_2, S, chain_M, chain_encoding_all, residue_idx,
                        mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np,
                        chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask,
                        pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi,
                        pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask,
                        pssm_bias_flag=bool(pssm_bias_flag),
                        tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta,
                        bias_by_res=bias_by_res_all
                    )
                    S_sample = sample_dict["S"]

                # score sampled sequences
                log_probs = model(
                    X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all,
                    randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"]
                )
                scores = _scores(S_sample, log_probs, mask_for_loss).cpu().data.numpy()

                # collect arrays
                all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
                all_log_probs_list.append(log_probs.cpu().data.numpy())
                S_sample_list.append(S_sample.cpu().data.numpy())

                # pretty print + saving
                for b_ix in range(BATCH_COPIES):
                    masked_chain_length_list = masked_chain_length_list_list[b_ix]
                    masked_list = masked_list_list[b_ix]

                    # recovery rate vs native
                    seq_recovery_rate = torch.sum(
                        torch.sum(torch.nn.functional.one_hot(S[b_ix], 21)
                                  * torch.nn.functional.one_hot(S_sample[b_ix], 21), axis=-1)
                        * mask_for_loss[b_ix]
                    ) / torch.sum(mask_for_loss[b_ix])

                    # sequences (split by chain)
                    seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
                    native_seq = _S_to_seq(S[b_ix], chain_M[b_ix])

                    # reorder by masked_list to print with separators
                    def _split_by_chains(seq_str):
                        start = 0; out = []; end = 0
                        for mask_l in masked_chain_length_list:
                            end += mask_l
                            out.append(seq_str[start:end])
                            start = end
                        return out

                    if b_ix == 0 and j == 0 and temp == temperatures[0]:
                        # print native first
                        list_of_AAs = _split_by_chains(native_seq)
                        native_seq_sorted = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                        l0 = 0
                        for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                            l0 += mc_length
                            native_seq_sorted = native_seq_sorted[:l0] + '/' + native_seq_sorted[l0:]
                            l0 += 1
                        sorted_masked_chain_letters = np.argsort(masked_list_list[0])
                        print_masked_chains = [masked_list_list[0][i] for i in sorted_masked_chain_letters]
                        sorted_visible_chain_letters = np.argsort(visible_list_list[0])
                        print_visible_chains = [visible_list_list[0][i] for i in sorted_visible_chain_letters]
                        native_score_print = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
                        line = f">{name_}, score={native_score_print}, fixed_chains={print_visible_chains}, designed_chains={print_masked_chains}, model_name={model_name}\n{native_seq_sorted}\n"
                        print(line.rstrip())

                    # sampled seq pretty formatting
                    list_of_AAs = _split_by_chains(seq)
                    seq_sorted = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                    l0 = 0
                    for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                        l0 += mc_length
                        seq_sorted = seq_sorted[:l0] + '/' + seq_sorted[l0:]
                        l0 += 1

                    score_print = np.format_float_positional(np.float32(scores[b_ix]), unique=False, precision=4)
                    seq_rec_print = np.format_float_positional(
                        np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4
                    )
                    header = f">T={temp}, sample={b_ix}, score={score_print}, seq_recovery={seq_rec_print}"
                    print(header)
                    print(seq_sorted)

                    # ---- 保存：FASTA + TSV（可选） ----
                    fasta_fh.write(f"{header}\n{seq_sorted.replace('/','')}\n")
                    tsv_fh.write(
                        f"{name_}\t{temp}\t{b_ix}\t{score_print}\t{seq_rec_print}\t{seq_sorted}\n"
                    )

        fasta_fh.close()  # ← 可选保存
        tsv_fh.close()    # ← 可选保存

# concat arrays (for downstream analysis)
all_probs_concat = np.concatenate(all_probs_list) if all_probs_list else np.zeros((0,))
all_log_probs_concat = np.concatenate(all_log_probs_list) if all_log_probs_list else np.zeros((0,))
S_sample_concat = np.concatenate(S_sample_list) if S_sample_list else np.zeros((0,))

print("Done.")
print("FASTA:", fasta_path)
print("TSV:  ", tsv_path)